# Project 1: Credit Risk Prediction (Tabular ML)

XGBoost + LightGBM ensemble on synthetic credit data.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, average_precision_score
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

X, y = make_classification(
    n_samples=50000, n_features=20, n_informative=15,
    n_redundant=3, n_clusters_per_class=2,
    weights=[0.85, 0.15], flip_y=0.01, random_state=42
)

feature_names = [
    "age","income","debt_ratio","credit_score","num_loans",
    "months_employed","num_credit_lines","balance","utilization",
    "missed_payments","loan_amount","interest_rate","collateral",
    "savings","monthly_expenses","dependents","years_at_address",
    "num_inquiries","total_assets","net_worth"
]
df = pd.DataFrame(X, columns=feature_names)
df["default"] = y
print("Dataset shape:", df.shape)
print("Default rate: {:.2f}%".format(df["default"].mean()*100))
print(df.describe().round(2).to_string())


Dataset shape: (50000, 21)
Default rate: 15.36%
            age    income  debt_ratio  credit_score  num_loans  months_employed  num_credit_lines   balance  utilization  missed_payments  loan_amount  interest_rate  collateral   savings  monthly_expenses  dependents  years_at_address  num_inquiries  total_assets  net_worth   default
count  50000.00  50000.00    50000.00      50000.00   50000.00         50000.00          50000.00  50000.00     50000.00         50000.00     50000.00       50000.00    50000.00  50000.00          50000.00    50000.00          50000.00       50000.00      50000.00   50000.00  50000.00
mean      -0.13     -0.00        0.60          0.01      -0.00             1.05              0.00      0.14         0.15            -2.01         0.00           0.15        0.15      0.70              0.86       -0.16             -0.85           0.00         -0.86      -0.86      0.15
std        2.34      1.00        4.44          2.47       1.00             5.55              2

In [2]:
X = df.drop("default", axis=1).values
y = df["default"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=6, eval_metric="logloss",
    random_state=42, verbosity=0
)
lgb_model = lgb.LGBMClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    class_weight="balanced", random_state=42, verbose=-1
)
lr_model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)

ensemble = VotingClassifier(
    estimators=[("xgb", xgb_model), ("lgb", lgb_model), ("lr", lr_model)],
    voting="soft", weights=[2, 2, 1]
)
ensemble.fit(X_train_s, y_train)
preds = ensemble.predict(X_test_s)
proba = ensemble.predict_proba(X_test_s)[:, 1]

print("=== Ensemble Results ===")
print(classification_report(y_test, preds, target_names=["No Default","Default"]))
print("ROC-AUC : {:.4f}".format(roc_auc_score(y_test, proba)))
print("PR-AUC  : {:.4f}".format(average_precision_score(y_test, proba)))
print("Confusion Matrix:")
print(confusion_matrix(y_test, preds))


=== Ensemble Results ===
              precision    recall  f1-score   support

  No Default       0.99      0.98      0.98      8464
     Default       0.88      0.95      0.91      1536

    accuracy                           0.97     10000
   macro avg       0.93      0.96      0.95     10000
weighted avg       0.97      0.97      0.97     10000

ROC-AUC : 0.9819
PR-AUC  : 0.9619
Confusion Matrix:
[[8263  201]
 [  80 1456]]


In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in [("XGBoost", xgb_model), ("LightGBM", lgb_model), ("LogReg", lr_model)]:
    scores = cross_val_score(model, X_train_s, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    print("{:10s}  AUC={:.4f} +/- {:.4f}".format(name, scores.mean(), scores.std()))

xgb_model.fit(X_train_s, y_train)
fi = sorted(zip(feature_names, xgb_model.feature_importances_), key=lambda x: -x[1])
print("\nTop 10 Features (XGBoost importance):")
for feat, imp in fi[:10]:
    bar = "#" * int(imp * 400)
    print("  {:20s} {:.4f}  {}".format(feat, imp, bar))


XGBoost     AUC=0.9832 +/- 0.0027


LightGBM    AUC=0.9818 +/- 0.0027


LogReg      AUC=0.9161 +/- 0.0013



Top 10 Features (XGBoost importance):
  missed_payments      0.1503  ############################################################
  age                  0.0970  ######################################
  savings              0.0954  ######################################
  monthly_expenses     0.0615  ########################
  collateral           0.0556  ######################
  utilization          0.0543  #####################
  years_at_address     0.0534  #####################
  interest_rate        0.0494  ###################
  num_credit_lines     0.0469  ##################
  dependents           0.0441  #################
